In [38]:
## pip install pandas numpy scikit-learn xgboost

## imports

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

import os

os.chdir("/Users/grace/Documents/GitHub/ml_vibe_check_ai_code_quality")

In [39]:
## file paths

TRAIN_PATH = "data/phase2/splits/train_features.csv"
VAL_PATH = "data/phase2/splits/val_features.csv"
TEST_PATH = "data/phase2/splits/test_features.csv"

# load data

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)

Train shape: (86398, 23)
Val shape:   (18508, 23)
Test shape:  (18510, 23)


In [40]:
# Quick look at the data
print("Train columns:")
print(train_df.columns.tolist())

Train columns:
['task_id', 'model_name', 'split', 'label', 'entry_point', 'libs', 'meta_parse_error', 'classical_loc', 'classical_cyclomatic_complexity', 'classical_max_nesting_depth', 'ast_if_count', 'ast_for_count', 'ast_while_count', 'ast_try_count', 'ast_except_count', 'ast_return_count', 'ast_import_count', 'ast_has_error_handling', 'align_lib_coverage', 'align_missing_libs', 'align_length_ratio', 'smell_hardcoded_return_funcs', 'smell_is_very_short']


In [41]:
# define features and target

meta_cols = ["task_id", "model_name", "split", "label", "entry_point", "libs"]

feature_cols = [
    "meta_parse_error",
    "classical_loc",
    "classical_cyclomatic_complexity",
    "classical_max_nesting_depth",
    "ast_if_count",
    "ast_for_count",
    "ast_while_count",
    "ast_try_count",
    "ast_except_count",
    "ast_return_count",
    "ast_import_count",
    "ast_has_error_handling",
    "align_lib_coverage",
    "align_missing_libs",
    "align_length_ratio",
    "smell_hardcoded_return_funcs",
    "smell_is_very_short",
]

target_col = "label"

In [42]:
X_train = train_df[feature_cols].copy()
y_train = train_df["label"].astype(int)
groups_train = train_df["task_id"]

# correlation with label

In [43]:
## correlation with label

feature_corr = (
    train_df[feature_cols]
    .corrwith(train_df[target_col])
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

print("\n=== FEATURE CORRELATION WITH LABEL ===")
print(feature_corr)


=== FEATURE CORRELATION WITH LABEL ===
classical_loc                     -0.174294
ast_has_error_handling            -0.121856
classical_cyclomatic_complexity   -0.117851
ast_try_count                     -0.115990
ast_except_count                  -0.112503
classical_max_nesting_depth       -0.104173
ast_import_count                  -0.086333
ast_for_count                     -0.075275
smell_is_very_short               -0.066709
ast_if_count                      -0.057220
ast_return_count                  -0.043487
smell_hardcoded_return_funcs      -0.010189
ast_while_count                    0.008300
align_lib_coverage                -0.006779
align_length_ratio                -0.002101
align_missing_libs                 0.001701
meta_parse_error                        NaN
dtype: float64


/Users/grace/miniforge3/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/grace/miniforge3/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Interpretation of Feature Correlations

The strongest predictors of failure are classical software metrics such as lines of code, cyclomatic complexity, and nesting depth, as well as structural indicators such as the presence of error handling and exception blocks. All of these features show negative correlation with the pass label, indicating that more complex solutions are less likely to be correct.

This suggests that LLM-generated code struggles primarily with task complexity. As solutions become longer and require more branching logic or error handling, models are more likely to produce incorrect outputs. This aligns with prior findings that LLMs exhibit “generation drift,” where later tokens are influenced more by previously generated code than by the original prompt.

Interestingly, prompt-code alignment features (e.g., library coverage and missing libraries) show near-zero correlation with correctness. This may indicate that simple heuristic alignment metrics are insufficient to capture whether generated code truly satisfies the prompt, or that alignment effects are nonlinear and better captured by more flexible models such as tree-based methods.

# Prepare X and y

In [44]:
X_train = train_df[feature_cols]
y_train = train_df[target_col].astype(int)

X_val = val_df[feature_cols]
y_val = val_df[target_col].astype(int)

X_test = test_df[feature_cols]
y_test = test_df[target_col].astype(int)

# Logistic Regression pipeline

In [45]:
log_reg_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        ),
    ]
)

# Random Forest pipeline

In [46]:
rf_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# XGBoost pipeline

In [47]:
xgb_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            XGBClassifier(
                n_estimators=400,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# Create evaluation function

In [48]:
def evaluate_model(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)

    # ===== TRAIN =====
    train_prob = model.predict_proba(X_train)[:, 1]
    train_pred = (train_prob >= 0.5).astype(int)

    train_acc = accuracy_score(y_train, train_pred)
    train_f1 = f1_score(y_train, train_pred)
    train_auc = roc_auc_score(y_train, train_prob)

    print("\n=== TRAIN RESULTS ===")
    print("Accuracy:", round(train_acc, 4))
    print("F1:", round(train_f1, 4))
    print("AUC:", round(train_auc, 4))

    # ===== VALIDATION =====
    val_prob = model.predict_proba(X_val)[:, 1]
    val_pred = (val_prob >= 0.5).astype(int)

    val_acc = accuracy_score(y_val, val_pred)
    val_f1 = f1_score(y_val, val_pred)
    val_auc = roc_auc_score(y_val, val_prob)

    print("\n=== VAL RESULTS ===")
    print("Accuracy:", round(val_acc, 4))
    print("F1:", round(val_f1, 4))
    print("AUC:", round(val_auc, 4))

    return {"train_auc": train_auc, "val_auc": val_auc}

# Train and evaluate

In [49]:
# Train and evaluate Logistic Regression

print("\n" + "=" * 60)
print("LOGISTIC REGRESSION")
print("=" * 60)

log_val_results = evaluate_model(log_reg_pipeline, X_train, y_train, X_val, y_val)


LOGISTIC REGRESSION

=== TRAIN RESULTS ===
Accuracy: 0.5833
F1: 0.5706
AUC: 0.6379

=== VAL RESULTS ===
Accuracy: 0.5827
F1: 0.596
AUC: 0.6368


In [50]:
# test and evaluate random forest

print("\n" + "=" * 60)
print("RANDOM FOREST")
print("=" * 60)

rf_val_results = evaluate_model(rf_pipeline, X_train, y_train, X_val, y_val)


RANDOM FOREST

=== TRAIN RESULTS ===
Accuracy: 0.9116
F1: 0.8934
AUC: 0.9755

=== VAL RESULTS ===
Accuracy: 0.5708
F1: 0.4331
AUC: 0.5841


In [51]:
# xgboost

print("\n" + "=" * 60)
print("XGBOOST")
print("=" * 60)

xgb_val_results = evaluate_model(xgb_pipeline, X_train, y_train, X_val, y_val)


XGBOOST

=== TRAIN RESULTS ===
Accuracy: 0.7174
F1: 0.5966
AUC: 0.7942

=== VAL RESULTS ===
Accuracy: 0.5769
F1: 0.3958
AUC: 0.6086


The initial training results show that logistic regression performs consistently, with very similar AUC on the training (0.6379) and validation sets (0.6368), indicating good generalization and little to no overfitting. In contrast, XGBoost achieves a much higher training AUC (0.7942) but drops substantially on validation (0.6086), suggesting significant overfitting. Random forest underperforms overall, with lower validation AUC (0.5841) and F1 score(0.4331), indicating weaker predictive power. Overall, the results suggest that simpler linear models better capture the relationship between the engineered features and code correctness, while more complex models struggle to generalize.

# hyperparameter tuning

In [52]:
## logisc regression tune
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"],
}

grid = GridSearchCV(
    estimator=log_reg_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=sgkf,
    n_jobs=-1,
)

grid.fit(X_train, y_train, groups=groups_train)

print("Best params:", grid.best_params_)
print("Best CV AUC:", grid.best_score_)

Best params: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV AUC: 0.630823941275491


In [53]:
## refit the best model and evaluate it on validation set again:

best_log_model = grid.best_estimator_

val_prob = best_log_model.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= 0.5).astype(int)

print("Validation AUC:", roc_auc_score(y_val, val_prob))
print("Validation F1:", f1_score(y_val, val_pred))
print("Validation Accuracy:", accuracy_score(y_val, val_pred))

Validation AUC: 0.6368153248927049
Validation F1: 0.5957290903381137
Validation Accuracy: 0.5826669548303436


In [54]:
from sklearn.metrics import classification_report

# predict probabilities
log_val_prob = log_reg_pipeline.predict_proba(X_val)[:, 1]

# convert to labels using threshold 0.5
log_val_pred = (log_val_prob >= 0.5).astype(int)

print("\n=== LOGISTIC REGRESSION VALIDATION REPORT ===")
print(classification_report(y_val, log_val_pred, digits=4))


=== LOGISTIC REGRESSION VALIDATION REPORT ===
              precision    recall  f1-score   support

           0     0.6641    0.4970    0.5685     10238
           1     0.5252    0.6889    0.5960      8270

    accuracy                         0.5827     18508
   macro avg     0.5947    0.5929    0.5823     18508
weighted avg     0.6021    0.5827    0.5808     18508



## Hyperparameter Tuning Results for logistic 

We performed hyperparameter tuning on logistic regression using grid search over the regularization parameter. The best parameter was found to be C=0.01, indicating slightly stronger regularization.

However, the tuned model achieved nearly identical performance to the default configuration, with validation AUC remaining at approximately 0.637. This suggests that the model is already well-calibrated and that further improvements are limited by the predictive power of the engineered features rather than model configuration.

# Tuning for xgboost

In [55]:
# hyperparameter tuning for xgboost

from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from scipy.stats import randint, uniform

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

xgb_param_dist = {
    "model__n_estimators": randint(100, 401),
    "model__max_depth": randint(2, 6),
    "model__learning_rate": uniform(0.01, 0.09),
    "model__subsample": uniform(0.6, 0.3),
    "model__colsample_bytree": uniform(0.6, 0.3),
    "model__min_child_weight": randint(1, 8),
    "model__gamma": uniform(0, 3),
    "model__reg_lambda": uniform(1, 5),
    "model__reg_alpha": uniform(0, 2),
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=sgkf,
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

xgb_search.fit(X_train, y_train, groups=groups_train)

print("Best params:", xgb_search.best_params_)
print("Best CV AUC:", xgb_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END model__colsample_bytree=0.7123620356542087, model__gamma=2.8521429192297485, model__learning_rate=0.07587945476302645, model__max_depth=2, model__min_child_weight=5, model__n_estimators=202, model__reg_alpha=0.8916655057071823, model__reg_lambda=1.4998745790900143, model__subsample=0.7377746675897601; total time=   0.8s
[CV] END model__colsample_bytree=0.7123620356542087, model__gamma=2.8521429192297485, model__learning_rate=0.07587945476302645, model__max_depth=2, model__min_child_weight=5, model__n_estimators=202, model__reg_alpha=0.8916655057071823, model__reg_lambda=1.4998745790900143, model__subsample=0.7377746675897601; total time=   0.8s
[CV] END model__colsample_bytree=0.7123620356542087, model__gamma=2.8521429192297485, model__learning_rate=0.07587945476302645, model__max_depth=2, model__min_child_weight=5, model__n_estimators=202, model__reg_alpha=0.8916655057071823, model__reg_lambda=1.4998745790900143, m

In [56]:
print("Best params:", xgb_search.best_params_)
print("Best CV AUC:", xgb_search.best_score_)

Best params: {'model__colsample_bytree': np.float64(0.8744879026631343), 'model__gamma': np.float64(2.550115733369398), 'model__learning_rate': np.float64(0.050450560672438305), 'model__max_depth': 2, 'model__min_child_weight': 7, 'model__n_estimators': 161, 'model__reg_alpha': np.float64(0.6503666440534941), 'model__reg_lambda': np.float64(4.64803089169032), 'model__subsample': np.float64(0.7912672414065639)}
Best CV AUC: 0.6336902612368912


In [57]:
best_xgb_model = xgb_search.best_estimator_

val_prob = best_xgb_model.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= 0.5).astype(int)

print("\n=== TUNED XGBOOST VALIDATION RESULTS ===")
print("Validation AUC:", roc_auc_score(y_val, val_prob))
print("Validation F1:", f1_score(y_val, val_pred))
print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print(classification_report(y_val, val_pred, digits=4))


=== TUNED XGBOOST VALIDATION RESULTS ===
Validation AUC: 0.6395331733520921
Validation F1: 0.329004329004329
Validation Accuracy: 0.5812621569051221
              precision    recall  f1-score   support

           0     0.5817    0.8652    0.6957     10238
           1     0.5793    0.2297    0.3290      8270

    accuracy                         0.5813     18508
   macro avg     0.5805    0.5475    0.5123     18508
weighted avg     0.5806    0.5813    0.5318     18508



In [58]:
train_prob = best_xgb_model.predict_proba(X_train)[:, 1]
train_pred = (train_prob >= 0.5).astype(int)

print("\n=== TUNED XGBOOST TRAIN RESULTS ===")
print("Train AUC:", roc_auc_score(y_train, train_prob))
print("Train F1:", f1_score(y_train, train_pred))
print("Train Accuracy:", accuracy_score(y_train, train_pred))


=== TUNED XGBOOST TRAIN RESULTS ===
Train AUC: 0.6635660563312089
Train F1: 0.37397214131447026
Train Accuracy: 0.6343086645524202


At the class level, logistic regression achieved a recall of 0.6152 for the positive class (correct solutions), significantly outperforming XGBoost, which achieved a recall of only 0.2297. This indicates that logistic regression is substantially more effective at identifying correct code outputs. Although precision for the positive class was lower (0.4743), the higher recall leads to a stronger F1 score (0.5356), making logistic regression more suitable for applications where detecting correct solutions is important.

## choose logistic as final model

We selected logistic regression as the final model because it provided the best balance between predictive performance, stability, and interpretability. On the validation set, logistic regression achieved an AUC of 0.6368 and the highest F1 score of 0.5957, outperforming both random forest (AUC = 0.5841, F1 = 0.4331) and XGBoost (AUC = 0.6086, F1 = 0.3958). Although tuned XGBoost slightly improved AUC to 0.6395, its F1 score remained much lower at 0.3290, driven by very low recall for the positive class (0.23), indicating poor classification performance without additional threshold tuning.

In contrast, logistic regression demonstrated strong generalization, with nearly identical training and validation AUC (0.6379 vs. 0.6368), showing no signs of overfitting. This stability carried over to the test set, where logistic regression achieved an AUC of 0.6083, F1 score of 0.5356, and accuracy of 0.5606, indicating consistent and reliable performance on unseen data.

Overall, logistic regression was chosen as the final model because it delivered the most balanced classification performance, required minimal tuning, and provided a more interpretable and stable solution compared to more complex models.

In [ ]:
# Retrain on full training data (train + val)

X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

best_log_model.fit(X_train_full, y_train_full)

,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [ ]:
# Final test evaluation

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    classification_report,
)

test_prob = best_log_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print("\n=== FINAL TEST RESULTS (LOGISTIC REGRESSION) ===")
print("AUC:", round(roc_auc_score(y_test, test_prob), 4))
print("F1:", round(f1_score(y_test, test_pred), 4))
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print(classification_report(y_test, test_pred, digits=4))


=== FINAL TEST RESULTS (LOGISTIC REGRESSION) ===
AUC: 0.6083
F1: 0.5356
Accuracy: 0.5606
              precision    recall  f1-score   support

           0     0.6596    0.5223    0.5830     10885
           1     0.4743    0.6152    0.5356      7625

    accuracy                         0.5606     18510
   macro avg     0.5669    0.5687    0.5593     18510
weighted avg     0.5832    0.5606    0.5635     18510



## Final Model Performance

The logistic regression model was evaluated on the held-out test set to obtain an unbiased estimate of performance. The model achieved an AUC of 0.6083, an F1 score of 0.5356, and an accuracy of 0.5606. These results indicate moderate predictive performance, suggesting that static code features provide some signal for predicting correctness but are insufficient for highly accurate classification.

At the class level, the model achieved higher recall for the positive class (0.6152), indicating that it is effective at identifying correct solutions, though at the cost of lower precision (0.4743). This reflects a tradeoff between identifying more correct outputs and maintaining prediction reliability.

Overall, the results suggest that while simple, interpretable features capture patterns, additional feature engineering or more expressive representations may be required to significantly improve performance.

### Real improvements would come from:

- Feature engineering (HIGH impact)
- interaction features (e.g. complexity × LOC)
- better alignment features
- semantic features